# 1-Implementar los endpoints

Haz una función en Python para cada uno de los endpoints del API REST:

- `GET /user/{email}`
- `GET /room/{room_id}`
- `GET /dungeon/{dungeon_id}`
- `POST /comment/`
- `DELETE /monster/{monster_id}`

Las funciones deben conectarse a la base de datos **MongoDB** y realizar las consultas pertinentes.

Se deben realizar todas las operaciones posibles directamente en la base de datos. No ejecutes cálculos en Python si no es necesario. 

In [2]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["jotuns_lair"]

rooms    = db["rooms"]
users    = db["users"]
loot     = db["loot"]
monsters = db["monsters"]

## Endpoints a implementar

#### **Información de un usuario** 
````GET /user/{email} ````

Este endpoint recibe el **email** de un usuario y devuelve toda la información disponible del mismo.

Además, incluye los **20 últimos comentarios** que ha realizado ese usuario.  
De cada comentario se debe mostrar:

- **Texto**
- **Fecha de creación**
- **Categoría**
- **Id de la habitación** (`Room.IdR`)
- **Nombre de la habitación** (`Room.name`)
- **Id de la mazmorra** (`Dungeon.IdD`)
- **Nombre de la mazmorra** (`Dungeon.name`)

> Los comentarios deben ordenarse por fecha de creación, mostrando primero los más recientes.

Como en este caso tenemos que ordenar un array de comentarios, podemos hacerlo de forma nativa en MongoDB usando sortArray. Te permite ordenar un array (en este caso, el array de comentarios) por un campo específico (en este caso, la fecha de creación). 

Para más información: https://www.mongodb.com/es/docs/manual/reference/operator/aggregation/sortArray/?msockid=0d7f1cc6f6d16b8426410bc2f76a6ae1

Nota: dentro del enunciado de la práctica, dentro de hints en users, la información de la habitación se llama references_room. Sin embargo, en los JSON de Moodle, se llama referemces_rooms. Mantenemos el nombre de los JSON de Moodle, es decir, references_rooms. room_Name en realidad se escribe como room_name. 

In [3]:
def get_user(email: str):
    """
    GET /user/{email}
    Devuelve todos los campos del usuario más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de room y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por email
        {"$match": {"email": email}},

        # Ordenar los hints por fecha desc y quedarse con los 20 últimos 
        {"$addFields": {
            "hints": {
                "$slice": [
                    {"$sortArray": {
                        "input": "$hints",
                        "sortBy": {"creation_date": -1}
                    }},
                    20
                ]
            }
        }},

        # Proyectar solo los campos necesarios
        {"$project": {
            "_id": 0,
            "email": 1,
            "user_name": 1,
            "creation_date": 1,
            "country": 1,
            "hints": {
                "text": 1,
                "creation_date": 1,
                "category": 1,
                "referemces_room.room_id": 1,
                "referemces_room.room_name": 1,
                "referemces_room.dungeon_id": 1,
                "referemces_room.dungeon_name": 1
            }
        }}
    ]

    result = list(users.aggregate(pipeline))
    return result[0] if result else None

In [4]:
get_user("abbottanne@example.com")

{'email': 'abbottanne@example.com',
 'hints': [{'text': 'Identify than professor statement support campaign computer.\\nEvent part half use plan. Already development front. Need today today set cold rock husband go.\\nWant safe PM front. Although born speech other project decision.\\nQuickly account staff imagine unit interest pick. Modern cultural someone appear rich quickly science. Director three red nice. True manager TV somebody school practice he.\\nRate size air off. Certain difficult drop walk cold share.\\nDraw standard prevent whole appear seat stand.',
   'category': 'lore',
   'creation_date': '2017-12-01 12:52:27.000000',
   'referemces_room': {'room_id': 667,
    'room_name': 'sticky sanctum of werewolves',
    'dungeon_id': 17,
    'dungeon_name': 'Marshgreat, Catacombs of the Wandering Degenerates'}},
  {'text': 'President result month range set specific magazine natural. Much agency bed himself production east interview. Involve yard line hear. Eat bed behind put proce

#### **Información de habitación en particular** 
````GET /room/{room_id} ````

Este endpoint recibe el **id de una habitación** y devuelve la siguiente información:

- **`idR`**
- **`name`**
- **`inWP`**
- **`outWP`**

Además, incluye:

- El **número de monstruos de cada tipo** presentes en la habitación.
- El **total de oro** que valen los tesoros de la sala.
- Los **últimos 20 comentarios** realizados sobre esa habitación.

Cada comentario debe incluir:

- **`userName`**
- **`country`**
- **`creationDate`** del usuario que lo realizó
- **Texto**
- **Fecha de publicación**
- **Categoría** del comentario

In [49]:
def get_room(room_id: int):
    """
    GET /room/{room_id}
    Devuelve todos los campos de la habitación más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de usuario y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por room_id
        {"$match": {"room_id": room_id}},

        {
            "$addFields": {
                "treasure_tot_value": {
                    "$sum": "$loot.gold"
                },
                "hint": {
                    "$slice": [
                        {"$sortArray": 
                         {
                            "input": "$hints",
                            "sortBy": {"creation_date": -1}
                        }
                        },
                        20
                    ]
            }
        }
        },

        {
            "$lookup": {
                "from": "monsters",
                "let": {"id_actual": "$_id"},
                "pipeline": [
                    {"$unwind": "$in_rooms"},
                    {"$match": {"in_rooms.room_id": room_id}},
                    
                    {"$group": {
                        "_id": "$type",
                        "count": {"$sum": "$in_rooms.amount"}
                    }},
                    
                    {"$project": {
                        "_id": 0,
                        "type": "$_id",
                        "count": 1
                    }}
                ],

                "as": "monsters_info"

            }
        },

        {"$project": {
            "_id": 0,
            "room_id": 1,
            "room_name": 1,
            "in_waypoint": 1,
            "out_waypoint": 1,
            "monsters_info": 1,
            "treasure_tot_value": 1,
            "hint": {
                "creation_date": 1,
                "hintText": 1,
                "category": 1,
                "publish_by": {
                    "user_name": 1,
                    "country": 1,
                    "creation_date": 1
                }

            },
        }
        }
    ]

    result = list(rooms.aggregate(pipeline))
    return result[0] if result else None

In [50]:
get_room(20)

{'room_id': 20,
 'room_name': 'lazy kitchen ',
 'in_waypoint': None,
 'out_waypoint': None,
 'treasure_tot_value': 17232.0,
 'hint': [{'category': 'suggestion',
   'hintText': 'Itaque officia placeat minus maxime. Facere eius vitae sequi. At neque nesciunt rerum fugit.\\nNesciunt in nemo omnis. Eligendi voluptatum accusamus.\\nError tempore suscipit quisquam. Soluta debitis architecto reprehenderit assumenda pariatur. Architecto aliquam a quibusdam blanditiis. Deleniti mollitia ab molestias ab mollitia.\\nNatus tenetur asperiores modi ab aliquam. Eligendi tempora sit alias magnam distinctio dolorum. Accusantium dolorem explicabo qui fugiat quaerat doloremque.',
   'publish_by': {'country': 'ko_KR',
    'user_name': 'sangho09',
    'creation_date': '2020-11-20'},
   'creation_date': '2022-08-10 16:11:20.000000'},
  {'category': 'hint',
   'hintText': "Froid lever à l'un. Hôtel pourtant lier heureux parole. Rester donc ne meilleur accent rassurer.\\nObéir courage habiller école changemen

#### **Información de una mazmorra** 
````GET /dungeon/{dungeon_id} ````

Este endpoint recibe el id de una mazmorra y devuelve información sobre una mazmorra del juego. 
Debe devolver: idM, name y lore. Además, este endpoint se utiliza para alimentar un grafo interactivo por lo que requiere la siguiente información: 

1) el nombre e id de cada habitación de la mazmorra; 
2) las conexiones entre habitaciones de la mazmorra; 
3) el id y el nombre de los monstruos que aparecen en cada habitación; 
4) el id y el nombre de los tesoros que aparecen en cada habitación; 
5) El número de comentarios de cada categoría que hay en cada habitación. 

NOTA: dentro de la base de datos, no tenemos un campo llamado lore. Por tanto, no se puede mostrar esa información.

NOTA: Interpretación del enunciado: como no se sabe si quiere las conexiones en un campo distinto o simplemente que lo tenga dentro de la habitación, entonces: 

{"$set": 
         {
             "rooms_conected": 
             {"$sortArray": 
              {
                  "input": {"$setUnion": 
                  [
                        ["$room_id"],
                        "$map": {
                            "input": "$rooms_connected",
                            "as": "room",
                            "in": "$room.room_id"
                        }
                  ]
                  },
                  "sortBy": 1
              }
             }
         }}


In [36]:
def get_dungeon(dungeon_id: int):
    """
    GET /dungeon/{dungeon_id}
    Devuelve todos los campos de la mazmorra más información del grafo interactivo.
    """

    pipeline = [
        {"$match": {"dungeon_id": dungeon_id}},
        
        {"$lookup": {
            "from": "rooms",
            "let": {"id_actual": "$_id"},
            "pipeline": [
                {"$match": {"dungeon_id": dungeon_id}},

                {
                    "$lookup": {
                        "from": "rooms",
                        "let": {"id_actual": "$_id"},
                        "pipeline": [
                            {"$match": {"dungeon_id": dungeon_id}},
                            {"$unwind": "$hints"},
                            {"$group": {
                                "_id": "$hints.category",
                                "total": {"$sum": 1}
                            }},
                            {"$project": {
                                "_id": 0,
                                "hint_category": "$_id",
                                "total": 1
                            }}
                        ],
                        "as": "hints_info"
                    }
                },

                {"$project": {
                "_id": 0,
                "room_id": 1,
                "room_name": 1,
                "rooms_connected": {
                    "$map": {
                        "input": "$rooms_connected",
                        "as": "r",
                        "in": {"room_id": "$$r.room_id"}
                    }
                },
                "monsters": {
                    "$map": {
                        "input": "$monsters",
                        "as": "m",
                        "in": {"id": "$$m.id", "name": "$$m.name"}
                    }
                },
                "loot": {
                    "$map": {
                        "input": {"$setUnion": "$loot"},
                        "as": "l",
                        "in": {"id": "$$l.id", "name": "$$l.name"}
                    }
                },
                "hints_info": 1
            }}
            ], 
            "as": "rooms_info"
        }},

        {"$project": {
            "_id": 0,
            "dungeon_id": 1,
            "dungeon_name": 1,
            "rooms_info": 1
        }}
        
    ]

    result = list(db["rooms"].aggregate(pipeline))
    return result[0] if result else None

In [37]:
from bson.objectid import ObjectId

get_dungeon(3)

{'dungeon_id': 3,
 'dungeon_name': 'Wanton, Culverts of the Jealous Thieves',
 'rooms_info': [{'room_id': 80,
   'room_name': 'sanctum ',
   'hints_info': [{'total': 19, 'hint_category': 'suggestion'},
    {'total': 186, 'hint_category': 'bug'},
    {'total': 385, 'hint_category': 'lore'},
    {'total': 178, 'hint_category': 'hint'}],
   'rooms_connected': [{'room_id': 79}, {'room_id': 81}, {'room_id': 108}],
   'monsters': None,
   'loot': None},
  {'room_id': 77,
   'room_name': 'unsightly dining room ',
   'hints_info': [{'total': 19, 'hint_category': 'suggestion'},
    {'total': 186, 'hint_category': 'bug'},
    {'total': 385, 'hint_category': 'lore'},
    {'total': 178, 'hint_category': 'hint'}],
   'rooms_connected': [{'room_id': 78}],
   'monsters': None,
   'loot': None},
  {'room_id': 81,
   'room_name': 'barracks ',
   'hints_info': [{'total': 19, 'hint_category': 'suggestion'},
    {'total': 186, 'hint_category': 'bug'},
    {'total': 385, 'hint_category': 'lore'},
    {'tot

### Información de una mazmorra  
`GET /dungeon/{dungeon_id}`

Este endpoint recibe el identificador de una mazmorra y devuelve información clave de una mazmorra del juego.

Debe incluir

- **Datos principales de la mazmorra**:
    - `idM`
    - `name`
    - `lore`

- **Datos para el grafo interactivo**:
    1. **Habitaciones de la mazmorra**: `id` y `name` de cada habitación.
    2. **Conexiones entre habitaciones** dentro de la mazmorra.
    3. **Monstruos por habitación**: `id` y `name` de cada monstruo.
    4. **Tesoros por habitación**: `id` y `name` de cada tesoro.
    5. **Comentarios por categoría en cada habitación**: número total por categoría.

#### **Publicar comentario** 
````POST /comment/```` 

Este endpoint añade un nuevo comentario. Recibe como parámetros: user_email (str), room_id (int), text (str), category (str).

In [39]:
def post_comment(user_email: str, room_id: int, text: str, category: str) -> dict:
    """POST /comment/: añade un comentario en el array hints de una habitación."""
    # Validar usuario
    user = users.find_one({"email": user_email})
    if not user:
        return {"ok": False, "error": "User not found"}

    # Construir comentario
    comment_room = {
        "hintText": text,
        "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "category": category,
        "publish_by": {
            "user_id": user.get("email"),
            "user_name": user.get("user_name"),
            "country": user.get("country"),
            "creation_date": user.get("creation_date")
        }
    }

    # Insertar en la habitación (push al array hints)
    result_room = rooms.update_one(
        {"room_id": room_id},
        {"$push": {"hints": comment_room}}
    )


    if result_room.matched_count == 0:
        return {"ok": False, "error": "Room not found"}

    if result_room.modified_count == 0:
        return {"ok": False, "error": "Comment was not inserted"}

    room = rooms.find_one({"room_id": room_id})

    result_user = users.update_one(
        {"email": user_email},
        {"$push": {"hints": {
            "text": text,
            "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "category": category,
            "referemces_room": {
                "room_id": room_id,
                "room_name": room.get("room_name"),
                "dungeon_id": room.get("dungeon_id"),
                "dungeon_name": room.get("dungeon_name")
            }
        }}})

    if result_user.matched_count == 0:
        return {"ok": False, "error": "User not found"}
    
    if result_user.modified_count == 0:
        return {"ok": False, "error": "Comment was not inserted in user"}


    return {
        "ok": True,
        "message": "Comment posted successfully",
        "room_id": room_id,
        "user_email": user_email
    }

In [40]:
# Prueba del endpoint POST /comment/
post_comment("abbottanne@example.com", 115, "prueba", "positive")

{'ok': True,
 'message': 'Comment posted successfully',
 'room_id': 115,
 'user_email': 'abbottanne@example.com'}

In [51]:
get_user("abbottanne@example.com")

{'email': 'abbottanne@example.com',
 'hints': [{'text': 'prueba',
   'creation_date': '2026-05-20 22:39:00',
   'category': 'positive',
   'referemces_room': {'room_id': 115,
    'room_name': 'sanctum sanctorum of degenerates',
    'dungeon_id': 3,
    'dungeon_name': 'Wanton, Culverts of the Jealous Thieves'}},
  {'text': 'Identify than professor statement support campaign computer.\\nEvent part half use plan. Already development front. Need today today set cold rock husband go.\\nWant safe PM front. Although born speech other project decision.\\nQuickly account staff imagine unit interest pick. Modern cultural someone appear rich quickly science. Director three red nice. True manager TV somebody school practice he.\\nRate size air off. Certain difficult drop walk cold share.\\nDraw standard prevent whole appear seat stand.',
   'category': 'lore',
   'creation_date': '2017-12-01 12:52:27.000000',
   'referemces_room': {'room_id': 667,
    'room_name': 'sticky sanctum of werewolves',


In [52]:
get_room(115)

{'room_id': 115,
 'room_name': 'sanctum sanctorum of degenerates',
 'in_waypoint': None,
 'out_waypoint': None,
 'treasure_tot_value': 0,
 'hint': [{'hintText': 'prueba',
   'creation_date': '2026-05-20 22:39:00',
   'category': 'positive',
   'publish_by': {'user_name': 'millermichael',
    'country': 'en_US',
    'creation_date': '2021-05-16'}},
  {'category': 'bug',
   'hintText': '当前开始发现电脑大小威望论坛.生产男人选择这些评论提高.\\n只要虽然觉得价格登录相关.当前是一质量制作责任.今天知道制作.\\n认为电子销售法律详细游戏.\\n国际标题网络特别系列一次.一样国家处理也是所有标准.得到最后不是数据上海价格发展.\\n网上比较原因最大作为.只有程序如果关系次数世界游戏.业务只有用户科技一个继续.\\n今年基本看到.部门文件发布公司结果网上通过.拥有非常信息地方工具.\\n单位不会希望个人比较只是发表.市场发表知道希望比较.法律文件一次文章投资今年.\\n处理评论电话觉得有些以上一种.免费感觉或者汽车制作活动知道学习.都是但是系列这些法律地方解决.应该空间因此无法.\\n记者不是投资之间详细.合作是一中心图片解决比较政府.以及更多介绍学生浏览很多分析全部.\\n欢迎完成如何其实.这里提高这是.\\n成为同时怎么出现设备.是否关于一般所以新闻最后公司.\\n其中国家资料那个.登录参加标题汽车由于功能.\\n功能科技电脑的是行业增加你的.支持提供自己拥有已经方式.解决历史可是使用.公司责任有关.\\n安全不能都是品牌音乐都是.技术男人他的产品手机经验非常生产.功能男人阅读位置一起现在进入.',
   'publish_by': {'country': 'zh_CN',
    'user_name': 'mingguo',
    'creation_date': '2022-0

#### **Borrar monstruo** 
````DELETE /monsters/{monster_id}```` 

Este endpoint recibe el id de un monstruo y lo elimina de la base de datos.

In [59]:
def delete_monster(monster_id: int) -> dict:
    """DELETE /monster/: elimina un monstruo de todas las habitaciones."""
    result_monster = monsters.delete_one({"id": monster_id})
    if result_monster.deleted_count == 0:
        return {"ok": False, "error": "Monster not found in monsters collection"}
    
    result_rooms = rooms.update_many(
        {"monsters.id": monster_id},
        {"$pull": {"monsters": {"id": monster_id}}}
    )

    if result_rooms.matched_count == 0:
        return {"ok": False, "error": "Monster not found in any room"}


    return {
        "ok": True,
        "message": f"Monster with id {monster_id} deleted from {result_rooms.modified_count} rooms"
    }

In [62]:
delete_monster(0)

{'ok': True, 'message': 'Monster with id 0 deleted from 17 rooms'}